**Important: read the following before you hit "Run all"**

Model checkpoints will be saved and loaded locally.

This notebook uses Google Colab. So, local means local disk used in the current Google Colab session.

To persist model checkpoints across sessions, this notebook saves the local copy to Google Drive after training.

To load a model back, this notebook copies model checkpoints from Google Drive back to Google Colab.

To transfer data between Google Colab and Google Drive, Google Drive is mounted under "/content/drive" and our data can be accessed via "/content/drive/MyDrive" in Google Colab.

**Final model saved after training:**

R1 on Google Drive: /checkpoints/r1_checkpoints/r1-sft-best

**Note**:

1) Change these locations and files operations to match your environments as necessary. For example, you may mount your Google Drive, then load and save checkpoints directly from Google Drive. Though, load & save on local + copy before load & save after training between Google Colab and Google Drive as described above is preferred.

2) Multiple checkpoints will be saved during training, which will consume a lot of disk space. Therefore, unless you want these checkpoints for resumption or continuous training across Google Colab sessions, you should only copy the final model from Google Colab to Google Drive. Otherwise, you may find out it would eat up your Google Drive space pretty fast.

In [ ]:
import os
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
)
from tqdm.auto import tqdm

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"

SEED = 42

MAX_TRAIN_EXAMPLES = 10_000
MAX_LENGTH = 128

BATCH_SIZE = 4
NUM_EPOCHS = 1

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

CHECKPOINT_EVERY_STEPS = 2_000

OUTPUT_DIR = Path("./r1_checkpoints")

MIN_WORDS = 8
MIN_PREFIX_WORDS = 4
MIN_COMPLETION_WORDS = 3

LABEL_TO_SENTIMENT = {
    0: "negative",
    1: "positive",
}


In [ ]:
# ============================================================
# 2. Reproducibility and device
# ============================================================

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)



Device: cuda


In [ ]:
# ============================================================
# 3. Load SST-2 and tokenizer
# ============================================================

sst2 = load_dataset("stanfordnlp/sst2")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# GPT-2 has no dedicated padding token.
tokenizer.pad_token = tokenizer.eos_token

PAD_TOKEN_ID = tokenizer.pad_token_id
EOS_TOKEN_ID = tokenizer.eos_token_id

print("Tokenizer vocabulary:", tokenizer.vocab_size)
print("PAD token ID:", PAD_TOKEN_ID)
print("EOS token ID:", EOS_TOKEN_ID)


Tokenizer vocabulary: 50257
PAD token ID: 50256
EOS token ID: 50256


In [ ]:
# ============================================================
# 4. Transform one SST-2 example
# ============================================================

def transform_example(example, rng):

    sentence = example["sentence"].strip()
    label = example["label"]

    words = sentence.split()

    if len(words) < MIN_WORDS:
        return None

    sentiment = LABEL_TO_SENTIMENT[label]

    split_ratio = rng.uniform(0.5, 0.7)
    split_index = int(len(words) * split_ratio)

    split_index = max(
        MIN_PREFIX_WORDS,
        split_index,
    )

    split_index = min(
        len(words) - MIN_COMPLETION_WORDS,
        split_index,
    )

    prefix = " ".join(words[:split_index])

    completion = " " + " ".join(words[split_index:])

    prompt = (
        f"Sentiment: {sentiment}\n"
        f"Continue the following review:\n"
        f"{prefix}"
    )

    return {
        "prompt": prompt,
        "completion": completion,
        "sentiment": sentiment,
    }

In [ ]:
# This cell is solely for the purpose of data inspection

# ---------------------------------------------------------
# Lazily generate transformed SST-2 examples for inspection
# ---------------------------------------------------------

def generate_dataset(dataset_split, seed=SEED):
    rng = random.Random(seed)

    for example in dataset_split:
        transformed = transform_example(example, rng)

        if transformed is not None:
            yield transformed


# --------------------------------------------------
# Load SST-2 for inspection
# --------------------------------------------------

sst2 = load_dataset("stanfordnlp/sst2")

train_dataset = generate_dataset(sst2["train"])
validation_dataset = generate_dataset(sst2["validation"])


# --------------------------------------------------
# Inspect the generated dataset
# --------------------------------------------------

# Get the first example.
example = next(train_dataset)

print(f"\n========== Example (1st) ==========")

print("\nPROMPT:")
print(example["prompt"])

print("\nCOMPLETION:")
print(example["completion"])

print("\nSENTIMENT:")
print(example["sentiment"])

# --------------------------------------------------
# Inspect 10 examples, create a fresh generator:
# --------------------------------------------------

from itertools import islice

train_dataset = generate_dataset(sst2["train"])

for i, example in enumerate(islice(train_dataset, 10)):
    print(f"\n========== Example {i + 1} ==========")
    print("PROMPT:", example["prompt"])
    print("COMPLETION:", example["completion"])

# --------------------------------------------------
# Count the examples without storing them
# --------------------------------------------------

train_count = sum(
    1 for _ in generate_dataset(sst2["train"])
)

validation_count = sum(
    1 for _ in generate_dataset(sst2["validation"])
)

print("\nTraining examples:", train_count)
print("\nValidation examples:", validation_count)

# --------------------------------------------------
# Inspect the sentiment distribution
# --------------------------------------------------


from collections import Counter

sentiment_counts = Counter(
    example["sentiment"]
    for example in generate_dataset(sst2["train"])
)

print(f"\nIn train dataset: {sentiment_counts}")

sentiment_counts = Counter(
    example["sentiment"]
    for example in generate_dataset(sst2["validation"])
)

print(f"\nIn validation dataset: {sentiment_counts}")


========== Example (1st) ==========

PROMPT:
Sentiment: positive
Continue the following review:
that loves its characters and communicates something

COMPLETION:
 rather beautiful about human nature

SENTIMENT:
positive

========== Example 1 ==========
PROMPT: Sentiment: positive
Continue the following review:
that loves its characters and communicates something
COMPLETION:  rather beautiful about human nature

========== Example 2 ==========
PROMPT: Sentiment: negative
Continue the following review:
remains utterly satisfied to
COMPLETION:  remain the same throughout

========== Example 3 ==========
PROMPT: Sentiment: negative
Continue the following review:
on the worst revenge-of-the-nerds clichés
COMPLETION:  the filmmakers could dredge up

========== Example 4 ==========
PROMPT: Sentiment: negative
Continue the following review:
that 's far too tragic
COMPLETION:  to merit such superficial treatment

========== Example 5 ==========
PROMPT: Sentiment: positive
Continue the followin

In [ ]:
# ============================================================
# 5. Lazy tokenized dataset
# ============================================================

class SST2ContinuationDataset(IterableDataset):

    def __init__(
        self,
        dataset_split,
        tokenizer,
        max_examples=None,
        shuffle=False,
        seed=SEED,
    ):
        self.dataset_split = dataset_split
        self.tokenizer = tokenizer
        self.max_examples = max_examples
        self.shuffle = shuffle
        self.seed = seed

    def __iter__(self):

        source = self.dataset_split

        if self.shuffle:
            source = source.shuffle(seed=self.seed)

        if self.max_examples is not None:
            count = min(
                self.max_examples,
                len(source),
            )
            source = source.select(range(count))

        rng = random.Random(self.seed)


        for example in source:

            transformed = transform_example(
                example,
                rng,
            )

            if transformed is None:
                continue

            prompt_ids = self.tokenizer.encode(
                transformed["prompt"],
                add_special_tokens=False,
            )

            completion_ids = self.tokenizer.encode(
                transformed["completion"],
                add_special_tokens=False,
            )


            # Reserve one position for EOS.
            if len(prompt_ids) >= MAX_LENGTH - 1:
                continue

            available_completion_length = (
                MAX_LENGTH - len(prompt_ids)
            )

            # Truncate completion if necessary,
            # always retaining EOS at the end.
            completion_ids = (
                completion_ids[:available_completion_length - 1]
                + [EOS_TOKEN_ID]
            )

            input_ids = prompt_ids + completion_ids


            # Prompt tokens do not contribute to SFT loss.
            labels = (
                [-100] * len(prompt_ids)
                + completion_ids
            )

            yield {
                "input_ids": input_ids,
                "labels": labels,
            }



In [ ]:

# ============================================================
# 6. Create lazy train and validation datasets
# ============================================================

train_dataset = SST2ContinuationDataset(
    dataset_split=sst2["train"],
    tokenizer=tokenizer,
    max_examples=MAX_TRAIN_EXAMPLES,
    shuffle=True,
    seed=SEED,
)

validation_dataset = SST2ContinuationDataset(
    dataset_split=sst2["validation"],
    tokenizer=tokenizer,
    shuffle=False,
    seed=SEED,
)

In [ ]:
# ============================================================
# 7. Dynamic padding collator
# ============================================================

def collate_fn(examples):

    max_length = max(
        len(example["input_ids"])
        for example in examples
    )

    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []


    for example in examples:

        input_ids = example["input_ids"]
        labels = example["labels"]

        padding_length = max_length - len(input_ids)

        batch_input_ids.append(
            input_ids + [PAD_TOKEN_ID] * padding_length
        )

        # Construct the mask from sequence lengths.
        # Do NOT infer it by comparing token IDs because
        # PAD and EOS intentionally have the same ID.
        batch_attention_mask.append(
            [1] * len(input_ids) + [0] * padding_length
        )

        batch_labels.append(
            labels + [-100] * padding_length
        )


    return {
        "input_ids": torch.tensor(
            batch_input_ids,
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            batch_attention_mask,
            dtype=torch.long,
        ),
        "labels": torch.tensor(
            batch_labels,
            dtype=torch.long,
        ),
    }


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=0,
)



In [ ]:
# ============================================================
# 8. Load GPT-2 causal language model
# ============================================================

model = GPT2LMHeadModel.from_pretrained(
    MODEL_NAME
)

model.config.pad_token_id = PAD_TOKEN_ID
model.config.use_cache = False

model.to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total parameters: 124439808
Trainable parameters: 124439808


In [ ]:

# ============================================================
# 9. Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


In [ ]:
# ============================================================
# 10. Checkpoint saving
# ============================================================

def save_checkpoint(
    model,
    tokenizer,
    checkpoint_dir,
    epoch,
    global_step,
    validation_loss=None,
):

    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Save model weights and model configuration.
    model.save_pretrained(checkpoint_dir)

    # Save the tokenizer alongside the model.
    tokenizer.save_pretrained(checkpoint_dir)

    # Save some experiment metadata.
    metadata = {
        "base_model": MODEL_NAME,
        "epoch": epoch,
        "global_step": global_step,
        "validation_loss": validation_loss,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "trainable_parameters": trainable_parameters,
    }


    with open(
        checkpoint_dir / "training_metadata.json",
        "w",
    ) as f:
        json.dump(metadata, f, indent=4)

    print(f"Checkpoint saved: {checkpoint_dir}")


In [ ]:
# ============================================================
# 11. Validation
# ============================================================

@torch.no_grad()
def evaluate(model, data_loader):

    model.eval()

    total_loss = 0.0
    total_target_tokens = 0

    for batch in data_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )


        # GPT2LMHeadModel shifts labels internally.
        # Position zero is not a prediction target.
        target_tokens = (
            labels[:, 1:] != -100
        ).sum().item()

        total_loss += outputs.loss.item() * target_tokens
        total_target_tokens += target_tokens

    average_loss = total_loss / total_target_tokens

    return average_loss



In [ ]:
# ============================================================
# 12. Training
# ============================================================

def train():

    global_step = 0
    best_validation_loss = float("inf")

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Determine the number of batches for progress reporting.
    # This does not materialize the transformed dataset.
    num_train_examples = min(
        MAX_TRAIN_EXAMPLES,
        len(sst2["train"]),
    )

    num_train_batches = (
        num_train_examples + BATCH_SIZE - 1
    ) // BATCH_SIZE


    for epoch in range(1, NUM_EPOCHS + 1):

        model.train()

        total_train_loss = 0.0
        total_target_tokens = 0

        print(f"\nStarting epoch {epoch}/{NUM_EPOCHS}")

        progress_bar = tqdm(
            train_loader,
            total=num_train_batches,
            desc=f"Epoch {epoch}/{NUM_EPOCHS}",
            unit="batch",
        )

        for batch in progress_bar:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)


            optimizer.zero_grad(set_to_none=True)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )

            optimizer.step()

            global_step += 1


            # Count actual completion tokens.
            target_tokens = (
                labels[:, 1:] != -100
            ).sum().item()

            total_train_loss += (
                loss.item() * target_tokens
            )

            total_target_tokens += target_tokens

            # Running average training loss.
            average_train_loss = (
                total_train_loss / total_target_tokens
            )

            current_lr = optimizer.param_groups[0]["lr"]


            # Update progress bar metrics.
            progress_bar.set_postfix(
                loss=f"{average_train_loss:.4f}",
                lr=f"{current_lr:.2e}",
                target_tokens=total_target_tokens,
                step=global_step,
            )

            # Periodically save model weights.
            if global_step % CHECKPOINT_EVERY_STEPS == 0:

                checkpoint_dir = (
                    OUTPUT_DIR
                    / f"checkpoint-step-{global_step}"
                )

                save_checkpoint(
                    model=model,
                    tokenizer=tokenizer,
                    checkpoint_dir=checkpoint_dir,
                    epoch=epoch,
                    global_step=global_step,
                )


        progress_bar.close()

        # ----------------------------------------
        # Epoch-level evaluation
        # ----------------------------------------

        average_train_loss = (
            total_train_loss / total_target_tokens
        )

        validation_loss = evaluate(
            model,
            validation_loader,
        )

        print(f"\nEpoch {epoch} completed")
        print(f"Training loss:   {average_train_loss:.4f}")
        print(f"Validation loss: {validation_loss:.4f}")


        # ----------------------------------------
        # Save best checkpoint
        # ----------------------------------------

        if validation_loss < best_validation_loss:

            best_validation_loss = validation_loss

            save_checkpoint(
                model=model,
                tokenizer=tokenizer,
                checkpoint_dir=OUTPUT_DIR / "r1-sft-best",
                epoch=epoch,
                global_step=global_step,
                validation_loss=validation_loss,
            )


        # ----------------------------------------
        # Save final checkpoint
        # ----------------------------------------

        save_checkpoint(
            model=model,
            tokenizer=tokenizer,
            checkpoint_dir=OUTPUT_DIR / "r1-sft-final",
            epoch=epoch,
            global_step=global_step,
            validation_loss=validation_loss,
        )

    print("\nR1 training finished.")
    print("Best validation loss:", best_validation_loss)

# ============================================================
# 13. Run training
# ============================================================

train()




Starting epoch 1/1


Epoch 1/1:   0%|          | 0/2500 [00:00<?, ?batch/s]


Epoch 1 completed
Training loss:   3.7584
Validation loss: 3.7565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved: r1_checkpoints/r1-sft-best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved: r1_checkpoints/r1-sft-final

R1 training finished.
Best validation loss: 3.7565116913603624


In [ ]:
# R1 generation

def generate_continuation(
    model,
    tokenizer,
    sentiment,
    prefix,
    max_new_tokens=40,
):

    prompt = (
        f"Sentiment: {sentiment}\n"
        f"Continue the following review:\n"
        f"{prefix}"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    model.eval()

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0, inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

In [ ]:
# Test R1 generation
prefix = "The restaurant was"

for sentiment in ["positive", "negative"]:

    continuation = generate_continuation(
        model,
        tokenizer,
        sentiment,
        prefix,
    )

    print(f"\nSentiment: {sentiment}")
    print(f"{prefix}{continuation}")


Sentiment: positive
The restaurant was like a mr. special --

Sentiment: negative
The restaurant was too dull and too boring


**Note**: Save a copy of saved model to a Google Drive folder

In [ ]:
# First, mount Drive (run in a cell)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Second, save r1 to Google Drive

# If we have enough storage to use on our Google Drive, save checkpoints and final model
!cp -r ./r1_checkpoints drive/MyDrive/checkpoints

# Otherwise, save final model
!cp -r ./r1_checkpoints/r1-sft-best drive/MyDrive/checkpoints/r1_checkpoints


In [ ]:
# Load & test the R1 model

from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
)

R1_MODEL_PATH = "./r1_checkpoints/r1-sft-best"

loaded_tokenizer = AutoTokenizer.from_pretrained(
    R1_MODEL_PATH
)

loaded_model = GPT2LMHeadModel.from_pretrained(
    R1_MODEL_PATH
)

loaded_model.eval()
loaded_model.to(device)
loaded_model.config.use_cache = True


prefix = "The restaurant was"

for sentiment in ["positive", "negative"]:

    continuation = generate_continuation(
        loaded_model,
        loaded_tokenizer,
        sentiment,
        prefix,
    )

    print(f"\nSentiment: {sentiment}")
    print(f"{prefix}{continuation}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Sentiment: positive
The restaurant was a must-see for americans who want something to do with the dead , but

Sentiment: negative
The restaurant was a cesspool
